# 1: Data Collection and Filtering

Goal: Build the analysis-ready train-delay dataset used by the later notebooks.

**Created files:**

- `data/processed/sbb_column_translation.csv`: English schema reference for the raw SBB columns.
- `data/processed/lausanne_station_variants.csv`: audit table showing which Lausanne-named stops appear in the sampled raw data.
- `data/processed/lausanne_arrivals_filtered.csv`: filtered train arrivals after the core inclusion rules are applied.
- `data/processed/lausanne_arrivals.csv`: engineered arrival-level table with the delay target and calendar features.
- `data/weather/lausanne_hourly_weather.csv`: hourly weather observations for the sampled Lausanne dates.
- `data/processed/dataset_final.csv`: final merged dataset used by `eda.ipynb` and `hypothesis_testing.ipynb`.

## 1.1 Setup and working directories

This section sets up the project root, the data folders, and the output folders so the rest of the notebook can read and write files consistently.

In [1]:
from __future__ import annotations

import csv
import io
import shutil
import subprocess
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
# Resolve the repository root whether the notebook is run inside notebooks/ or from the project root.
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
# Raw monthly extracts and their processed derivatives both live under data/.
DATA_DIR = ROOT / "data"
# Keep filtered CSV outputs with train rows only in a stable location for the downstream notebooks.
PROCESSED_DIR = DATA_DIR / "processed"
# Store the hourly Open-Meteo exports separately from the train tables.
WEATHER_DIR = DATA_DIR / "weather"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
WEATHER_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Data directory: {DATA_DIR.relative_to(ROOT)}")
print(f"Processed output dir: {PROCESSED_DIR.relative_to(ROOT)}")


Project root: /Users/berenaydogan/Documents/4.2/DSA210/Project/DSA210-Project
Data directory: data
Processed output dir: data/processed


## 1.2 Archive constants and date helpers

This section defines the sampled date ranges, archive naming rules, and the small helper functions that support the collection logic.

The sampled date ranges are:

- January 1 to January 30, 2025
- April 1 to April 30, 2025
- July 1 to July 30, 2025
- October 1 to October 30, 2025

I use one 30 day period from each season so the dataset covers winter, spring, summer, and autumn without requiring a full year of raw data.

In [2]:
import datetime as dt
import re
import time
import urllib.error
import urllib.parse
import urllib.request
import zipfile
from collections import defaultdict

ARCHIVE_INDEX_URL = "https://archive.opentransportdata.swiss/istdaten.php"
ARCHIVE_BASE_URL = "https://archive.opentransportdata.swiss/"
# Stream large monthly ZIP files in 1 MB chunks so the download can report progress.
CHUNK_SIZE = 1024 * 1024
# Use one shared timeout for every archive and weather HTTP request.
DOWNLOAD_TIMEOUT_SECONDS = 60
# Retry transient network failures a few times before giving up.
MAX_DOWNLOAD_RETRIES = 5
# Print progress periodically instead of on every chunk write.
PROGRESS_INTERVAL_SECONDS = 5
DEFAULT_SAMPLE_RANGES = (
    ("2025-01-01", "2025-01-30"),
    ("2025-04-01", "2025-04-30"),
    ("2025-07-01", "2025-07-30"),
    ("2025-10-01", "2025-10-30"),
)
# Find every monthly ZIP link published on the official archive index page.
ARCHIVE_LINK_RE = re.compile(r'href="(?P<href>istdaten/[^"]+\.zip)"')
# Match the modern archive naming scheme (for example 2025-01.zip).
ARCHIVE_MONTH_RE = re.compile(r"(?P<year>20\d{2})-(?P<month>\d{2})\.zip$")
# Match older short names that still appear in some archive listings.
ARCHIVE_MONTH_SHORT_RE = re.compile(r"(?P<yy>\d{2})_(?P<month>\d{2})\.zip$")
# Prefer the highest explicit archive version when multiple ZIPs exist for one month.
ARCHIVE_VERSION_RE = re.compile(r"-v(?P<version>\d+)-")


def parse_iso_date(raw: str) -> dt.date:
    """Convert a YYYY-MM-DD string into a Python date object."""
    return dt.date.fromisoformat(raw)


def expand_date_ranges(date_ranges: tuple[tuple[str, str], ...] = DEFAULT_SAMPLE_RANGES) -> list[dt.date]:
    """Expand inclusive date ranges like 2025-01-01..2025-01-30 into daily dates."""
    dates: list[dt.date] = []
    for raw_start, raw_end in date_ranges:
        start = parse_iso_date(raw_start)
        end = parse_iso_date(raw_end)
        if end < start:
            raise ValueError(f"Invalid range: {raw_start} > {raw_end}")
        current = start
        while current <= end:
            dates.append(current)
            current += dt.timedelta(days=1)
    return dates


def fetch_text(url: str) -> str:
    """Fetch a small UTF-8 text payload such as the archive index HTML page."""
    request = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; DSA210 notebook downloader)"},
    )
    with urllib.request.urlopen(request, timeout=DOWNLOAD_TIMEOUT_SECONDS) as response:
        return response.read().decode("utf-8")


def month_key_from_href(href: str) -> str | None:
    """Normalize an archive filename into a YYYY-MM key for month-level lookup."""
    name = Path(href).name
    match = ARCHIVE_MONTH_RE.search(name)
    if match:
        return f"{match.group('year')}-{match.group('month')}"

    match = ARCHIVE_MONTH_SHORT_RE.search(name)
    if match:
        return f"20{match.group('yy')}-{match.group('month')}"

    return None


def archive_version_rank(href: str) -> int:
    match = ARCHIVE_VERSION_RE.search(Path(href).name)
    if match:
        return int(match.group("version"))
    return 1


def parse_archive_index(html_text: str) -> dict[str, list[str]]:
    month_to_hrefs: dict[str, list[str]] = defaultdict(list)
    for match in ARCHIVE_LINK_RE.finditer(html_text):
        href = match.group("href")
        month_key = month_key_from_href(href)
        if month_key:
            month_to_hrefs[month_key].append(href)

    resolved: dict[str, list[str]] = {}
    for month_key, hrefs in month_to_hrefs.items():
        unique_hrefs = sorted(
            set(hrefs),
            key=lambda href: (archive_version_rank(href), href),
            reverse=True,
        )
        resolved[month_key] = [
            urllib.parse.urljoin(ARCHIVE_BASE_URL, href) for href in unique_hrefs
        ]
    return resolved


def group_dates_by_month(dates: list[dt.date]) -> dict[str, list[dt.date]]:
    grouped: dict[str, list[dt.date]] = defaultdict(list)
    for item in dates:
        grouped[item.strftime("%Y-%m")].append(item)
    return dict(grouped)


def format_bytes(num_bytes: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    value = float(num_bytes)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{int(value)} {unit}" if unit == "B" else f"{value:.1f} {unit}"
        value /= 1024
    return f"{num_bytes} B"


## 1.3 Archive index, download, and extraction helpers

This section collects the helper functions for finding the right monthly archives, downloading them when needed, and extracting only the daily CSV files that belong to the study period.

Implementation detail: safe to skim on a first read.

In [3]:
def head_file_size(url: str) -> int | None:
    request = urllib.request.Request(
        url,
        method="HEAD",
        headers={"User-Agent": "Mozilla/5.0 (compatible; DSA210 notebook downloader)"},
    )
    with urllib.request.urlopen(request, timeout=DOWNLOAD_TIMEOUT_SECONDS) as response:
        length = response.headers.get("Content-Length")
        return int(length) if length is not None else None


def print_progress(downloaded: int, total: int | None, start_time: float) -> None:
    elapsed = max(time.monotonic() - start_time, 1e-9)
    speed = downloaded / elapsed
    if total:
        pct = downloaded / total * 100
        print(
            f"    progress: {format_bytes(downloaded)} / {format_bytes(total)} ({pct:5.1f}%) at {format_bytes(int(speed))}/s",
            flush=True,
        )
    else:
        print(f"    progress: {format_bytes(downloaded)} at {format_bytes(int(speed))}/s", flush=True)


def download_file(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temp_path = destination.with_suffix(destination.suffix + ".part")
    total_size = head_file_size(url)
    existing_size = temp_path.stat().st_size if temp_path.exists() else 0

    if total_size is not None and existing_size >= total_size:
        temp_path.replace(destination)
        return

    if existing_size:
        print(f"  resuming partial archive at {format_bytes(existing_size)}", flush=True)
    else:
        size_text = format_bytes(total_size) if total_size else "unknown size"
        print(f"  starting archive download ({size_text})", flush=True)

    downloaded = existing_size
    start_time = time.monotonic()
    last_progress = start_time
    retries = 0

    while True:
        headers = {"User-Agent": "Mozilla/5.0 (compatible; DSA210 notebook downloader)"}
        if downloaded:
            headers["Range"] = f"bytes={downloaded}-"
        request = urllib.request.Request(url, headers=headers)

        try:
            with urllib.request.urlopen(request, timeout=DOWNLOAD_TIMEOUT_SECONDS) as response:
                status = response.getcode()
                # HTTP 206 confirms the server honored the resume request for a partial archive.
                if downloaded and status != 206:
                    temp_path.unlink(missing_ok=True)
                    downloaded = 0
                    print("  server did not honor resume request; restarting download", flush=True)
                    continue

                mode = "ab" if downloaded else "wb"
                with temp_path.open(mode) as fh:
                    while True:
                        chunk = response.read(CHUNK_SIZE)
                        if not chunk:
                            break
                        fh.write(chunk)
                        downloaded += len(chunk)
                        now = time.monotonic()
                        if now - last_progress >= PROGRESS_INTERVAL_SECONDS:
                            print_progress(downloaded, total_size, start_time)
                            last_progress = now

        except (TimeoutError, urllib.error.URLError) as exc:
            retries += 1
            if retries > MAX_DOWNLOAD_RETRIES:
                raise urllib.error.URLError(
                    f"{exc}. Download stopped after {MAX_DOWNLOAD_RETRIES} retries."
                ) from exc
            print(f"  download interrupted ({exc}); retrying {retries}/{MAX_DOWNLOAD_RETRIES}...", flush=True)
            time.sleep(min(2**retries, 30))
            continue

        break

    print_progress(downloaded, total_size, start_time)
    temp_path.replace(destination)


In [4]:
def find_zip_member(zf: zipfile.ZipFile, target_date: dt.date) -> str | None:
    target = target_date.isoformat()
    matches = []
    for member in zf.namelist():
        name = Path(member).name.lower()
        if target in name and name.endswith(".csv"):
            matches.append(member)
    if not matches:
        return None
    if len(matches) > 1:
        raise FileNotFoundError(
            f"Multiple CSV matches for {target} found inside {zf.filename}: {matches}"
        )
    return matches[0]


def output_path_for_date(output_dir: Path, target_date: dt.date) -> Path:
    month_dir = output_dir / target_date.strftime("%Y-%m")
    return month_dir / f"{target_date.isoformat()}_istdaten.csv"


def migrate_legacy_output(output_dir: Path, target_date: dt.date) -> Path | None:
    legacy_path = output_dir / f"{target_date.isoformat()}_istdaten.csv"
    new_path = output_path_for_date(output_dir, target_date)
    if not legacy_path.exists() or new_path.exists():
        return new_path if new_path.exists() else None
    new_path.parent.mkdir(parents=True, exist_ok=True)
    legacy_path.replace(new_path)
    print(f"  moved existing {legacy_path.name} -> {new_path.parent.name}/", flush=True)
    return new_path


def extract_member_with_unzip(archive_path: Path, member: str, output_path: Path) -> None:
    unzip_path = shutil.which("unzip")
    if unzip_path is None:
        raise RuntimeError(
            "Encountered an unsupported ZIP compression method and 'unzip' is not available."
        )
    with output_path.open("wb") as dst:
        result = subprocess.run(
            [unzip_path, "-p", str(archive_path), member],
            stdout=dst,
            stderr=subprocess.PIPE,
            check=False,
        )
    if result.returncode != 0:
        output_path.unlink(missing_ok=True)
        stderr = result.stderr.decode("utf-8", errors="replace").strip()
        raise RuntimeError(
            f"Failed to extract {member} from {archive_path.name} with unzip: {stderr}"
        )


def extract_member(
    zf: zipfile.ZipFile,
    member: str,
    target_date: dt.date,
    output_dir: Path,
    overwrite: bool,
) -> Path:
    migrated = migrate_legacy_output(output_dir, target_date)
    output_path = output_path_for_date(output_dir, target_date)
    if (output_path.exists() or migrated is not None) and not overwrite:
        return output_path

    output_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with zf.open(member) as src, output_path.open("wb") as dst:
            shutil.copyfileobj(src, dst, length=CHUNK_SIZE)
        return output_path
    except NotImplementedError:
        output_path.unlink(missing_ok=True)
        archive_path = Path(zf.filename) if zf.filename is not None else None
        if archive_path is None:
            raise
        extract_member_with_unzip(archive_path, member, output_path)
        return output_path


In [5]:
def download_selected_istdaten(
    date_ranges: tuple[tuple[str, str], ...] = DEFAULT_SAMPLE_RANGES,
    output_dir: Path = DATA_DIR,
    archive_dir: Path | None = None,
    keep_archives: bool = True,
    overwrite: bool = False,
    dry_run: bool = False,
) -> None:
    dates = expand_date_ranges(date_ranges)
    output_dir = output_dir.resolve()
    archive_dir = (archive_dir or (output_dir / "_archives")).resolve()

    archive_index = parse_archive_index(fetch_text(ARCHIVE_INDEX_URL))
    by_month = group_dates_by_month(dates)
    missing_months = [month for month in by_month if month not in archive_index]
    if missing_months:
        raise FileNotFoundError(
            "No archive ZIP found for these months in the official index: " + ", ".join(sorted(missing_months))
        )

    print(f"Requested {len(dates)} day(s) across {len(by_month)} month archive(s).")
    print(f"Output directory: {output_dir}")
    print(f"Archive directory: {archive_dir}")
    print("Extracted CSV layout: <output-dir>/<YYYY-MM>/<YYYY-MM-DD>_istdaten.csv")

    for month in sorted(by_month):
        archive_urls = archive_index[month]
        month_dates = by_month[month]
        print(f"\n[{month}]")
        print(f"  sources: {len(archive_urls)} archive(s) available")
        preview = ", ".join(date.isoformat() for date in month_dates[:3])
        print("  dates: " + preview + (" ..." if len(month_dates) > 3 else ""))

        if dry_run:
            for archive_url in archive_urls:
                print(f"  source: {archive_url}")
            continue

        remaining_dates = list(month_dates)
        used_archive_paths: list[Path] = []

        for archive_url in archive_urls:
            archive_name = Path(urllib.parse.urlparse(archive_url).path).name
            archive_path = archive_dir / archive_name
            used_archive_paths.append(archive_path)

            print(f"  source: {archive_url}")
            if archive_path.exists():
                print(f"    archive already exists: {archive_path}")
            else:
                print("    downloading archive...")
                download_file(archive_url, archive_path)

            extracted_any = False
            next_remaining: list[dt.date] = []
            with zipfile.ZipFile(archive_path) as zf:
                for target_date in remaining_dates:
                    member = find_zip_member(zf, target_date)
                    # Dates missing from the current ZIP stay in the queue for the next fallback archive.
                    if member is None:
                        next_remaining.append(target_date)
                        continue

                    migrated = migrate_legacy_output(output_dir, target_date)
                    output_path = output_path_for_date(output_dir, target_date)
                    if output_path.exists() and not overwrite:
                        print(f"    skip existing {output_path.parent.name}/{output_path.name}")
                        continue
                    if migrated is not None and not overwrite:
                        print(f"    skip migrated {output_path.parent.name}/{output_path.name}")
                        continue

                    extract_member(zf, member, target_date, output_dir, overwrite)
                    print(f"    extracted {output_path.parent.name}/{output_path.name}")
                    extracted_any = True

            remaining_dates = next_remaining
            if not remaining_dates:
                break
            if not extracted_any:
                print("    archive did not contain requested early dates; trying fallback archive", flush=True)

        if remaining_dates:
            missing = ", ".join(date.isoformat() for date in remaining_dates)
            raise FileNotFoundError(
                f"No archive for {month} contained these requested dates: {missing}"
            )

        if not keep_archives:
            for archive_path in used_archive_paths:
                if archive_path.exists():
                    archive_path.unlink()
                    print(f"  removed archive after extraction: {archive_path.name}")


## 1.4 Raw download configuration

This section decides whether the notebook should reuse the raw daily CSV files that are already present or download the selected months again from the archive.

In [6]:
# Leave this False once the daily CSV extracts already exist under data/YYYY-MM/.
RUN_RAW_DOWNLOAD = False
# The default sample covers four 30-day windows so the project spans multiple seasons.
RAW_DATE_RANGES = DEFAULT_SAMPLE_RANGES
# Keep the downloaded monthly ZIP files unless disk space is more important than reruns.
KEEP_ARCHIVES = True
# Only set this True if you intentionally want to replace previously extracted daily CSVs.
OVERWRITE_RAW_FILES = False

expected_raw_files = expand_date_ranges(RAW_DATE_RANGES)
existing_raw_files = sorted(DATA_DIR.glob("2025-*/*.csv"))
print(f"Existing extracted daily CSV files: {len(existing_raw_files)}")
print(f"Expected files for the default sample: {len(expected_raw_files)}")

if RUN_RAW_DOWNLOAD:
    download_selected_istdaten(
        date_ranges=RAW_DATE_RANGES,
        output_dir=DATA_DIR,
        keep_archives=KEEP_ARCHIVES,
        overwrite=OVERWRITE_RAW_FILES,
    )
else:
    print("Skipping raw data download. Set RUN_RAW_DOWNLOAD = True if the data/YYYY-MM/ folders are missing.")

RAW_FILES = sorted(DATA_DIR.glob("2025-*/*.csv"))
# Stop early with a clear message instead of letting the later filtering steps fail cryptically.
assert RAW_FILES, "No extracted CSV files found under data/YYYY-MM/. Set RUN_RAW_DOWNLOAD = True to download them."

RG_PATH = shutil.which("rg")
if RG_PATH is None:
    print(
        "ripgrep was not found. Falling back to pure Python row scanning. "
        "This is slower, but it produces the same downstream filtered dataset."
    )
else:
    print(f"Using ripgrep for faster preliminary filtering: {RG_PATH}")

with RAW_FILES[0].open("r", encoding="utf-8-sig") as fh:
    HEADER_LINE = fh.readline()

print(f"Daily CSV files found: {len(RAW_FILES)}")
print(f"First file: {RAW_FILES[0].relative_to(ROOT)}")


Existing extracted daily CSV files: 120
Expected files for the default sample: 120
Skipping raw data download. Set RUN_RAW_DOWNLOAD = True if the data/YYYY-MM/ folders are missing.
ripgrep was not found. Falling back to pure Python row scanning. This is slower, but it produces the same downstream filtered dataset.
Daily CSV files found: 120
First file: data/2025-01/2025-01-01_istdaten.csv


## 1.5 Translation map

This section defines English names for all raw columns used in the source SBB files. I keep the full translated raw schema at this stage.

**Files produced in next step:** `data/processed/sbb_column_translation.csv`


In [7]:
# Represent the translation table as a list of dictionaries so one structure can drive the documentation and column renaming.
column_map = [
    {"raw_column": "BETRIEBSTAG", "english_name": "service_date", "description": "Service date"},
    {"raw_column": "FAHRT_BEZEICHNER", "english_name": "trip_id", "description": "Journey identifier"},
    {"raw_column": "BETREIBER_ID", "english_name": "operator_id", "description": "Operator identifier"},
    {"raw_column": "BETREIBER_ABK", "english_name": "operator_abbr", "description": "Operator abbreviation"},
    {"raw_column": "BETREIBER_NAME", "english_name": "operator_name", "description": "Operator name"},
    {"raw_column": "PRODUKT_ID", "english_name": "product_type", "description": "Product type, e.g. train"},
    {"raw_column": "LINIEN_ID", "english_name": "line_id", "description": "Line identifier"},
    {"raw_column": "LINIEN_TEXT", "english_name": "line_name", "description": "Line text label"},
    {"raw_column": "UMLAUF_ID", "english_name": "circulation_id", "description": "Circulation/run identifier"},
    {"raw_column": "VERKEHRSMITTEL_TEXT", "english_name": "vehicle_text", "description": "Vehicle text label"},
    {"raw_column": "ZUSATZFAHRT_TF", "english_name": "is_extra_trip", "description": "Extra trip flag"},
    {"raw_column": "FAELLT_AUS_TF", "english_name": "is_cancelled", "description": "Cancellation flag"},
    {"raw_column": "BPUIC", "english_name": "stop_uic", "description": "Stop UIC/BPUIC code"},
    {"raw_column": "HALTESTELLEN_NAME", "english_name": "stop_name", "description": "Stop name"},
    {"raw_column": "ANKUNFTSZEIT", "english_name": "scheduled_arrival", "description": "Scheduled arrival timestamp"},
    {"raw_column": "AN_PROGNOSE", "english_name": "predicted_arrival", "description": "Predicted or actual arrival timestamp"},
    {"raw_column": "AN_PROGNOSE_STATUS", "english_name": "arrival_prediction_status", "description": "Arrival prediction status"},
    {"raw_column": "ABFAHRTSZEIT", "english_name": "scheduled_departure", "description": "Scheduled departure timestamp"},
    {"raw_column": "AB_PROGNOSE", "english_name": "predicted_departure", "description": "Predicted or actual departure timestamp"},
    {"raw_column": "AB_PROGNOSE_STATUS", "english_name": "departure_prediction_status", "description": "Departure prediction status"},
    {"raw_column": "DURCHFAHRT_TF", "english_name": "is_pass_through", "description": "Pass-through flag"},
]

raw_header_columns = [column.strip() for column in HEADER_LINE.strip().split(";")]
translated_raw_columns = [item["raw_column"] for item in column_map]
assert raw_header_columns == translated_raw_columns, (
    "The translation map should cover all raw columns in the same order as the source CSV header."
)

translation_df = pd.DataFrame(column_map)
translation_path = PROCESSED_DIR / "sbb_column_translation.csv"
translation_df.to_csv(translation_path, index=False)
display(translation_df)
print(f"Saved translation map to {translation_path.relative_to(ROOT)}")


,raw_column,english_name,description
0,BETRIEBSTAG,service_date,Service date
1,FAHRT_BEZEICHNER,trip_id,Journey identifier
2,BETREIBER_ID,operator_id,Operator identifier
3,BETREIBER_ABK,operator_abbr,Operator abbreviation
4,BETREIBER_NAME,operator_name,Operator name
5,PRODUKT_ID,product_type,"Product type, e.g. train"
6,LINIEN_ID,line_id,Line identifier
7,LINIEN_TEXT,line_name,Line text label
8,UMLAUF_ID,circulation_id,Circulation/run identifier
9,VERKEHRSMITTEL_TEXT,vehicle_text,Vehicle text label


Saved translation map to data/processed/sbb_column_translation.csv


## 1.6 Station scope review

This section checks at which stops with Lausanne in the name appear in the raw sample and how many candidate arrival rows are available before the stricter filtering begins.

In [8]:
def rg_lausanne_lines(csv_path: Path) -> str:
    """Return only raw lines containing the literal string 'Lausanne'."""
    result = subprocess.run(
        [RG_PATH, "--fixed-strings", "--no-filename", "--color", "never", "Lausanne", str(csv_path)],
        capture_output=True,
        text=True,
        check=False,
    )
    # ripgrep returns 1 when no matches are found, so only codes outside {0, 1} are true failures.
    if result.returncode not in (0, 1):
        raise RuntimeError(
            f"rg failed for {csv_path.name}: {result.stderr.strip() or result.returncode}"
        )
    return result.stdout


def iter_candidate_rows_python(csv_path: Path):
    """Yield candidate rows directly from the CSV when ripgrep is unavailable."""
    with csv_path.open("r", encoding="utf-8-sig", newline="") as fh:
        reader = csv.DictReader(fh, delimiter=";")
        for row in reader:
            if "Lausanne" in row.get("HALTESTELLEN_NAME", ""):
                yield row


def iter_candidate_rows(csv_path: Path):
    if RG_PATH is None:
        yield from iter_candidate_rows_python(csv_path)
        return

    candidate_text = rg_lausanne_lines(csv_path)
    if not candidate_text:
        return
    # Prepend the original CSV header so DictReader can parse rg output that contains only matching rows as valid rows.
    reader = csv.DictReader(io.StringIO(HEADER_LINE + candidate_text), delimiter=";")
    for row in reader:
        yield row


def has_text(value: str | None) -> bool:
    return bool((value or "").strip())


def is_lausanne_train_arrival(row: dict[str, str]) -> bool:
    return (
        row.get("PRODUKT_ID") == "Zug"
        and "Lausanne" in row.get("HALTESTELLEN_NAME", "")
        and has_text(row.get("ANKUNFTSZEIT"))
    )


def is_usable_lausanne_arrival(row: dict[str, str]) -> bool:
    return (
        is_lausanne_train_arrival(row)
        and has_text(row.get("AN_PROGNOSE"))
        and row.get("FAELLT_AUS_TF", "").lower() != "true"
    )

**Files produced in next step:** `data/processed/lausanne_station_variants.csv`


In [9]:
# Counter tracks reusable audit metrics, while station_counts retains row totals for each stop for inspection.
station_metrics = Counter()
station_counts = Counter()
lausanne_scope_rows = []

for csv_path in RAW_FILES:
    for row in iter_candidate_rows(csv_path) or []:
        if not is_lausanne_train_arrival(row):
            continue

        # Cache the broader Lausanne arrival scope once so later cells can reuse the same row set.
        lausanne_scope_rows.append(
            {
                "row": row,
                "source_month": csv_path.parent.name,
                "source_file": csv_path.name,
            }
        )

        stop_name = row["HALTESTELLEN_NAME"]
        station_metrics["train_arrival_rows"] += 1
        station_counts[stop_name] += 1

        if has_text(row.get("AN_PROGNOSE")):
            station_metrics["train_arrival_rows_with_prediction"] += 1
        if row.get("FAELLT_AUS_TF", "").lower() != "true":
            station_metrics["train_arrival_rows_not_cancelled"] += 1
        if is_usable_lausanne_arrival(row):
            station_metrics["usable_delay_rows"] += 1

station_scope_df = (
    pd.DataFrame(
        [{"stop_name": stop_name, "arrival_rows": rows} for stop_name, rows in station_counts.items()]
    )
    .sort_values("arrival_rows", ascending=False)
    .reset_index(drop=True)
)

print("Station review across the 120 selected days:")
print(f"  train arrival rows at stops whose names contain Lausanne: {station_metrics['train_arrival_rows']:,}")
print(f"  with non-empty AN_PROGNOSE: {station_metrics['train_arrival_rows_with_prediction']:,}")
print(f"  not cancelled: {station_metrics['train_arrival_rows_not_cancelled']:,}")
print(f"  usable for later delay labeling: {station_metrics['usable_delay_rows']:,}")
print(f"  average usable rows per selected day: {station_metrics['usable_delay_rows'] / len(RAW_FILES):.1f}")
display(station_scope_df)

station_scope_df.to_csv(PROCESSED_DIR / "lausanne_station_variants.csv", index=False)

Station review across the 120 selected days:
  train arrival rows at stops whose names contain Lausanne: 96,158
  with non-empty AN_PROGNOSE: 90,818
  not cancelled: 92,738
  usable for later delay labeling: 90,818
  average usable rows per selected day: 756.8


,stop_name,arrival_rows
0,Lausanne,67558
1,Romanel-sur-Lausanne,11440
2,Lausanne-Chauderon,11440
3,Lausanne-Flon,5720


## 1.7 Filtered Lausanne arrivals

This section filters the train arrivals in the Lausanne area, omitting the cancelled ones and the ones without the time fields needed for delay analysis.

In [10]:
rename_map = {item["raw_column"]: item["english_name"] for item in column_map}
kept_columns = list(rename_map.keys())
records = []

for scoped_entry in lausanne_scope_rows:
    row = scoped_entry["row"]
    if not is_usable_lausanne_arrival(row):
        continue

    kept_row = {key: row.get(key, "") for key in kept_columns}
    # Preserve the original source month so the final merge can recover the intended seasonal window label.
    kept_row["source_month"] = scoped_entry["source_month"]
    kept_row["source_file"] = scoped_entry["source_file"]
    records.append(kept_row)

**Files produced in next step:** `data/processed/lausanne_arrivals_filtered.csv`


In [11]:
filtered_df = pd.DataFrame.from_records(records).rename(columns=rename_map)

bool_cols = ["is_extra_trip", "is_cancelled", "is_pass_through"]
for col in bool_cols:
    # Normalize lowercase string flags into real booleans for cleaner downstream summaries and modeling.
    filtered_df[col] = filtered_df[col].map({"true": True, "false": False})

filtered_df["service_date"] = pd.to_datetime(filtered_df["service_date"], dayfirst=True, errors="coerce")
for col in ["scheduled_arrival", "predicted_arrival", "scheduled_departure", "predicted_departure"]:
    filtered_df[col] = pd.to_datetime(filtered_df[col], dayfirst=True, errors="coerce")

filtered_df = filtered_df.sort_values(["scheduled_arrival", "trip_id", "stop_name"]).reset_index(drop=True)

output_path = PROCESSED_DIR / "lausanne_arrivals_filtered.csv"
filtered_df.to_csv(output_path, index=False)

summary_df = (
    filtered_df["stop_name"]
    .value_counts()
    .rename_axis("stop_name")
    .reset_index(name="rows")
)

print(f"Saved filtered dataset to {output_path.relative_to(ROOT)}")
print(f"Filtered rows: {len(filtered_df):,}")
print(f"Unique trips: {filtered_df['trip_id'].nunique():,}")
print(f"Average rows per selected day: {len(filtered_df) / len(RAW_FILES):.1f}")
display(summary_df)
display(filtered_df.head())


Saved filtered dataset to data/processed/lausanne_arrivals_filtered.csv
Filtered rows: 90,818
Unique trips: 1,699
Average rows per selected day: 756.8


,stop_name,rows
0,Lausanne,63942
1,Romanel-sur-Lausanne,10749
2,Lausanne-Chauderon,10749
3,Lausanne-Flon,5378


,service_date,trip_id,operator_id,operator_abbr,operator_name,product_type,line_id,line_name,circulation_id,vehicle_text,...,stop_name,scheduled_arrival,predicted_arrival,arrival_prediction_status,scheduled_departure,predicted_departure,departure_prediction_status,is_pass_through,source_month,source_file
0,2025-01-01,ch:1:sjyid:100001:13832-001,85:11,SBB,Schweizerische Bundesbahnen SBB,Zug,13832,SN,,SN,...,Lausanne,2025-01-01 01:25:00,2025-01-01 01:24:48,REAL,2025-01-01 01:27:00,2025-01-01 01:27:22,REAL,False,2025-01,2025-01-01_istdaten.csv
1,2025-01-01,ch:1:sjyid:100001:13831-001,85:11,SBB,Schweizerische Bundesbahnen SBB,Zug,13831,SN,,SN,...,Lausanne,2025-01-01 01:27:00,2025-01-01 01:28:00,REAL,NaT,NaT,,False,2025-01,2025-01-01_istdaten.csv
2,2025-01-01,ch:1:sjyid:100001:13836-001,85:11,SBB,Schweizerische Bundesbahnen SBB,Zug,13836,SN,,SN,...,Lausanne,2025-01-01 01:34:00,2025-01-01 01:35:18,REAL,NaT,NaT,,False,2025-01,2025-01-01_istdaten.csv
3,2025-01-01,ch:1:sjyid:100001:13835-001,85:11,SBB,Schweizerische Bundesbahnen SBB,Zug,13835,SN,,SN,...,Lausanne,2025-01-01 01:35:00,2025-01-01 01:34:21,REAL,2025-01-01 01:35:00,2025-01-01 01:39:37,REAL,False,2025-01,2025-01-01_istdaten.csv
4,2025-01-01,ch:1:sjyid:100001:13837-001,85:11,SBB,Schweizerische Bundesbahnen SBB,Zug,13837,SN,,SN,...,Lausanne,2025-01-01 01:52:00,2025-01-01 01:52:11,REAL,NaT,NaT,,False,2025-01,2025-01-01_istdaten.csv


## 1.8 Feature selection and engineering

This section filters the columns of the filtered arrival table and creates the delay target, and the core time features.

In [12]:
filtered_input_path = PROCESSED_DIR / "lausanne_arrivals_filtered.csv"
engineered_output_path = PROCESSED_DIR / "lausanne_arrivals.csv"

engineered_df = pd.read_csv(
    filtered_input_path,
    parse_dates=[
        "service_date",
        "scheduled_arrival",
        "predicted_arrival",
        "scheduled_departure",
        "predicted_departure",
    ],
)

# Delay is measured as actual/predicted arrival minus scheduled arrival, expressed in minutes.
engineered_df["delay_minutes"] = (
    engineered_df["predicted_arrival"] - engineered_df["scheduled_arrival"]
).dt.total_seconds().div(60)
# Treat any positive delay as the positive class used in the later hypothesis and ML notebooks.
engineered_df["delayed"] = (engineered_df["delay_minutes"] > 0).astype(int)
engineered_df["scheduled_arrival_hour"] = engineered_df["scheduled_arrival"].dt.hour
engineered_df["scheduled_arrival_weekday"] = engineered_df["scheduled_arrival"].dt.dayofweek
# Keep the weekday name alongside the numeric code so plots and models can use easy to read categories.
engineered_df["scheduled_arrival_weekday_name"] = engineered_df["scheduled_arrival"].dt.day_name()
engineered_df["scheduled_arrival_month"] = engineered_df["scheduled_arrival"].dt.month
engineered_df["is_weekend"] = engineered_df["scheduled_arrival_weekday"].isin([5, 6])

required_feature_columns = [
    "product_type",
    "line_id",
    "line_name",
    "operator_id",
    "operator_abbr",
    "operator_name",
    "stop_name",
]
missing_feature_columns = [
    col for col in required_feature_columns if col not in engineered_df.columns
]
# Verify that the service descriptors needed for EDA and modeling survived the filtering stage.
assert not missing_feature_columns, (
    f"Missing expected service-related columns: {missing_feature_columns}"
)
# After filtering out missing predicted arrivals, every engineered row should yield a valid numeric delay.
assert engineered_df["delay_minutes"].notna().all(), (
    "delay_minutes should be complete after the arrival filters."
)

model_input_candidates = [
    "scheduled_arrival_hour",
    "scheduled_arrival_weekday",
    "scheduled_arrival_weekday_name",
    "scheduled_arrival_month",
    "is_weekend",
    "product_type",
    "line_id",
    "line_name",
    "operator_id",
    "operator_abbr",
    "operator_name",
    "stop_name",
]


**Files produced in next step:** `data/processed/lausanne_arrivals.csv`


In [13]:
engineered_df = engineered_df.sort_values(
    ["scheduled_arrival", "trip_id", "stop_name"]
).reset_index(drop=True)
engineered_df.to_csv(engineered_output_path, index=False)

delay_summary = pd.DataFrame(
    {
        "metric": [
            "rows",
            "late arrivals",
            "delay rate",
            "mean delay minutes",
            "median delay minutes",
            "min delay minutes",
            "max delay minutes",
        ],
        "value": [
            len(engineered_df),
            int(engineered_df["delayed"].sum()),
            engineered_df["delayed"].mean(),
            engineered_df["delay_minutes"].mean(),
            engineered_df["delay_minutes"].median(),
            engineered_df["delay_minutes"].min(),
            engineered_df["delay_minutes"].max(),
        ],
    }
)

print(f"Saved engineered dataset to {engineered_output_path.relative_to(ROOT)}")
print(
    "Predicted/actual arrival remains in the file for target construction and reference analysis only."
)
print("Candidate model input columns:")
for column in model_input_candidates:
    print(f"- {column}")

display(delay_summary)
display(
    engineered_df[
        [
            "scheduled_arrival",
            "predicted_arrival",
            "delay_minutes",
            "delayed",
            "scheduled_arrival_hour",
            "scheduled_arrival_weekday_name",
            "scheduled_arrival_month",
            "is_weekend",
            "line_name",
            "operator_abbr",
            "stop_name",
        ]
    ].head()
)


Saved engineered dataset to data/processed/lausanne_arrivals.csv
Predicted/actual arrival remains in the file for target construction and reference analysis only.
Candidate model input columns:
- scheduled_arrival_hour
- scheduled_arrival_weekday
- scheduled_arrival_weekday_name
- scheduled_arrival_month
- is_weekend
- product_type
- line_id
- line_name
- operator_id
- operator_abbr
- operator_name
- stop_name


,metric,value
0,rows,90818.000000
1,late arrivals,43058.000000
2,delay rate,0.474113
3,mean delay minutes,0.362635
4,median delay minutes,-0.066667
5,min delay minutes,-147.350000
6,max delay minutes,119.933333


,scheduled_arrival,predicted_arrival,delay_minutes,delayed,scheduled_arrival_hour,scheduled_arrival_weekday_name,scheduled_arrival_month,is_weekend,line_name,operator_abbr,stop_name
0,2025-01-01 01:25:00,2025-01-01 01:24:48,-0.200000,0,1,Wednesday,1,False,SN,SBB,Lausanne
1,2025-01-01 01:27:00,2025-01-01 01:28:00,1.000000,1,1,Wednesday,1,False,SN,SBB,Lausanne
2,2025-01-01 01:34:00,2025-01-01 01:35:18,1.300000,1,1,Wednesday,1,False,SN,SBB,Lausanne
3,2025-01-01 01:35:00,2025-01-01 01:34:21,-0.650000,0,1,Wednesday,1,False,SN,SBB,Lausanne
4,2025-01-01 01:52:00,2025-01-01 01:52:11,0.183333,1,1,Wednesday,1,False,SN,SBB,Lausanne


## 1.9 Hourly weather collection and export

This section collects hourly weather for Lausanne so each scheduled arrival can later be matched to the weather conditions for the corresponding hour.

In [14]:
import json

GEOCODING_API_URL = "https://geocoding-api.open-meteo.com/v1/search"
WEATHER_ARCHIVE_API_URL = "https://archive-api.open-meteo.com/v1/archive"
WEATHER_VARIABLES = ["temperature_2m", "precipitation", "snowfall", "wind_speed_10m"]
# Match the local Swiss timezone used by train schedules so hourly weather keys line up without manual offsets.
WEATHER_TIMEZONE = "Europe/Zurich"
weather_output_path = WEATHER_DIR / "lausanne_hourly_weather.csv"


def fetch_json(url: str, params: dict[str, object]) -> dict:
    request = urllib.request.Request(
        f"{url}?{urllib.parse.urlencode(params, doseq=True)}",
        headers={"User-Agent": "Mozilla/5.0 (compatible; DSA210 weather collector)"},
    )
    with urllib.request.urlopen(request, timeout=DOWNLOAD_TIMEOUT_SECONDS) as response:
        return json.load(response)


In [15]:
# Geocode Lausanne once so every weather request uses the same latitude/longitude pair.
geo_response = fetch_json(
    GEOCODING_API_URL,
    {
        "name": "Lausanne",
        "count": 1,
        "language": "en",
        "format": "json",
        "countryCode": "CH",
    },
)
assert geo_response.get("results"), "Open-Meteo geocoding did not return a Lausanne result."

lausanne_geo = geo_response["results"][0]
assert lausanne_geo["timezone"] == WEATHER_TIMEZONE, (
    f"Expected Lausanne timezone {WEATHER_TIMEZONE}, got {lausanne_geo['timezone']}"
)

weather_query_ranges = [
    (
        start_date,
        # Extend the end date by one day so arrivals late at night and just after midnight still find an hourly match.
        (parse_iso_date(end_date) + dt.timedelta(days=1)).isoformat(),
    )
    for start_date, end_date in RAW_DATE_RANGES
]


In [16]:
weather_frames = []
weather_units = None

for start_date, end_date in weather_query_ranges:
    weather_response = fetch_json(
        WEATHER_ARCHIVE_API_URL,
        {
            "latitude": lausanne_geo["latitude"],
            "longitude": lausanne_geo["longitude"],
            "start_date": start_date,
            "end_date": end_date,
            "hourly": ",".join(WEATHER_VARIABLES),
            "timezone": WEATHER_TIMEZONE,
        },
    )

    hourly_df = pd.DataFrame(weather_response["hourly"]).rename(
        columns={"time": "weather_time"}
    )
    hourly_df["weather_time"] = pd.to_datetime(hourly_df["weather_time"])
    hourly_df["weather_date"] = hourly_df["weather_time"].dt.strftime("%Y-%m-%d")
    hourly_df["weather_hour"] = hourly_df["weather_time"].dt.hour
    hourly_df["request_start_date"] = start_date
    hourly_df["request_end_date"] = end_date
    # Concatenate one request frame per seasonal window, then deduplicate by timestamp at the end.
    weather_frames.append(hourly_df)

    if weather_units is None:
        weather_units = weather_response["hourly_units"]

weather_df = pd.concat(weather_frames, ignore_index=True)
weather_df = weather_df.sort_values("weather_time").drop_duplicates(
    subset=["weather_time"]
).reset_index(drop=True)

weather_df["location_name"] = lausanne_geo["name"]
weather_df["country"] = lausanne_geo["country"]
weather_df["admin1"] = lausanne_geo.get("admin1", "")
weather_df["timezone"] = WEATHER_TIMEZONE
weather_df["latitude"] = lausanne_geo["latitude"]
weather_df["longitude"] = lausanne_geo["longitude"]

assert weather_df["weather_time"].notna().all(), "Weather timestamps should be complete."
assert not weather_df[["weather_date", "weather_hour"]].duplicated().any(), (
    "Weather date/hour merge keys should be unique."
)

expected_weather_rows = len(expand_date_ranges(weather_query_ranges)) * 24


**Files produced in next step:** `data/weather/lausanne_hourly_weather.csv`


In [17]:
weather_df.to_csv(weather_output_path, index=False)

units_df = pd.DataFrame(
    {
        "variable": list(weather_units.keys()),
        "unit": list(weather_units.values()),
    }
)
weather_counts = (
    weather_df["weather_time"]
    .dt.strftime("%Y-%m")
    .value_counts()
    .sort_index()
    .rename_axis("month")
    .reset_index(name="hours")
)

print(
    f"Resolved Lausanne to {lausanne_geo['latitude']}, {lausanne_geo['longitude']} ({lausanne_geo['admin1']}, {lausanne_geo['country']})"
)
print(f"Saved weather dataset to {weather_output_path.relative_to(ROOT)}")
print(f"Hourly weather rows: {len(weather_df):,} (expected {expected_weather_rows:,})")

display(units_df)
display(weather_counts)
display(weather_df.head())


Resolved Lausanne to 46.516, 6.63282 (Vaud, Switzerland)
Saved weather dataset to data/weather/lausanne_hourly_weather.csv
Hourly weather rows: 2,976 (expected 2,976)


,variable,unit
0,time,iso8601
1,temperature_2m,°C
2,precipitation,mm
3,snowfall,cm
4,wind_speed_10m,km/h


,month,hours
0,2025-01,744
1,2025-04,720
2,2025-05,24
3,2025-07,744
4,2025-10,744


,weather_time,temperature_2m,precipitation,snowfall,wind_speed_10m,weather_date,weather_hour,request_start_date,request_end_date,location_name,country,admin1,timezone,latitude,longitude
0,2025-01-01 00:00:00,3.1,0.0,0.0,6.8,2025-01-01,0,2025-01-01,2025-01-31,Lausanne,Switzerland,Vaud,Europe/Zurich,46.516,6.63282
1,2025-01-01 01:00:00,2.6,0.0,0.0,6.2,2025-01-01,1,2025-01-01,2025-01-31,Lausanne,Switzerland,Vaud,Europe/Zurich,46.516,6.63282
2,2025-01-01 02:00:00,2.6,0.0,0.0,7.0,2025-01-01,2,2025-01-01,2025-01-31,Lausanne,Switzerland,Vaud,Europe/Zurich,46.516,6.63282
3,2025-01-01 03:00:00,0.7,0.0,0.0,8.7,2025-01-01,3,2025-01-01,2025-01-31,Lausanne,Switzerland,Vaud,Europe/Zurich,46.516,6.63282
4,2025-01-01 04:00:00,0.7,0.0,0.0,8.5,2025-01-01,4,2025-01-01,2025-01-31,Lausanne,Switzerland,Vaud,Europe/Zurich,46.516,6.63282


## 1.10 Final merge 

This section joins the train arrival table with the hourly weather table, keeps the columns used later in the analysis, and writes the final merged dataset.

In [18]:
final_output_path = PROCESSED_DIR / "dataset_final.csv"

# Collapse the sampled source months into seasonal window labels that are easy to use in the report.
season_map = {
    "2025-01": "Winter (Jan window)",
    "2025-04": "Spring (Apr window)",
    "2025-07": "Summer (Jul window)",
    "2025-10": "Autumn (Oct window)",
}

final_train_df = pd.read_csv(
    engineered_output_path,
    parse_dates=["service_date", "scheduled_arrival"],
)
final_weather_df = pd.read_csv(weather_output_path, parse_dates=["weather_time"])

final_train_df["scheduled_arrival_date"] = final_train_df["scheduled_arrival"].dt.strftime("%Y-%m-%d")

merged_df = final_train_df.merge(
    final_weather_df,
    left_on=["scheduled_arrival_date", "scheduled_arrival_hour"],
    right_on=["weather_date", "weather_hour"],
    how="left",
    # Each train row can map to one hourly weather record, but many train rows may share that same hour.
    validate="many_to_one",
    indicator=True,
)

unmatched_rows = merged_df[merged_df["_merge"] != "both"].copy()
merge_summary = pd.DataFrame(
    {
        "metric": [
            "train rows",
            "merged rows",
            "unmatched weather rows",
            "merge coverage",
            "late arrivals",
            "delay rate",
        ],
        "value": [
            len(final_train_df),
            len(merged_df),
            len(unmatched_rows),
            1 - (len(unmatched_rows) / len(merged_df)),
            int(merged_df["delayed"].sum()),
            merged_df["delayed"].mean(),
        ],
    }
)

merged_df["weather_matched"] = merged_df["_merge"] == "both"
merged_df["season_window"] = merged_df["source_month"].map(season_map)
merged_df["train_type"] = merged_df["vehicle_text"]
merged_df["operator"] = merged_df["operator_abbr"]

# This is where the notebook narrows the merged table down to the prepared for analysis fields used by EDA, hypothesis testing, and the later modeling work.
final_columns = [
    "service_date",
    "season_window",
    "scheduled_arrival",
    "scheduled_arrival_date",
    "delay_minutes",
    "delayed",
    "scheduled_arrival_hour",
    "scheduled_arrival_weekday",
    "scheduled_arrival_weekday_name",
    "scheduled_arrival_month",
    "is_weekend",
    "stop_name",
    "train_type",
    "line_name",
    "operator",
    "weather_time",
    "temperature_2m",
    "precipitation",
    "snowfall",
    "wind_speed_10m",
    "weather_matched",
]

assert merged_df["season_window"].notna().all(), "Every row should map to a season window."
assert merged_df["train_type"].notna().all(), "Every row should have a train type."
# Downstream notebooks assume complete weather coverage, so fail fast if any train rows missed the hourly join.
assert unmatched_rows.empty, "Every train arrival should match exactly one hourly weather record."


**Files produced in next step:** `data/processed/dataset_final.csv`


In [19]:
dataset_final = merged_df[final_columns].sort_values(
    ["scheduled_arrival", "stop_name"]
).reset_index(drop=True)
dataset_final.to_csv(final_output_path, index=False)

print(f"Saved final merged dataset to {final_output_path.relative_to(ROOT)}")
print(f"Final analysis columns: {len(final_columns)}")
print(f"Unmatched train rows after merge: {len(unmatched_rows):,}")
print(f"Merge coverage: {(1 - (len(unmatched_rows) / len(dataset_final))):.4%}")

display(merge_summary)
if len(unmatched_rows):
    display(
        unmatched_rows[
            [
                "service_date",
                "scheduled_arrival",
                "scheduled_arrival_date",
                "scheduled_arrival_hour",
                "stop_name",
                "source_file",
            ]
        ].head(10)
    )
display(dataset_final.head())


Saved final merged dataset to data/processed/dataset_final.csv
Final analysis columns: 21
Unmatched train rows after merge: 0
Merge coverage: 100.0000%


,metric,value
0,train rows,90818.000000
1,merged rows,90818.000000
2,unmatched weather rows,0.000000
3,merge coverage,1.000000
4,late arrivals,43058.000000
5,delay rate,0.474113


,service_date,season_window,scheduled_arrival,scheduled_arrival_date,delay_minutes,delayed,scheduled_arrival_hour,scheduled_arrival_weekday,scheduled_arrival_weekday_name,scheduled_arrival_month,...,stop_name,train_type,line_name,operator,weather_time,temperature_2m,precipitation,snowfall,wind_speed_10m,weather_matched
0,2025-01-01,Winter (Jan window),2025-01-01 01:25:00,2025-01-01,-0.200000,0,1,2,Wednesday,1,...,Lausanne,SN,SN,SBB,2025-01-01 01:00:00,2.6,0.0,0.0,6.2,True
1,2025-01-01,Winter (Jan window),2025-01-01 01:27:00,2025-01-01,1.000000,1,1,2,Wednesday,1,...,Lausanne,SN,SN,SBB,2025-01-01 01:00:00,2.6,0.0,0.0,6.2,True
2,2025-01-01,Winter (Jan window),2025-01-01 01:34:00,2025-01-01,1.300000,1,1,2,Wednesday,1,...,Lausanne,SN,SN,SBB,2025-01-01 01:00:00,2.6,0.0,0.0,6.2,True
3,2025-01-01,Winter (Jan window),2025-01-01 01:35:00,2025-01-01,-0.650000,0,1,2,Wednesday,1,...,Lausanne,SN,SN,SBB,2025-01-01 01:00:00,2.6,0.0,0.0,6.2,True
4,2025-01-01,Winter (Jan window),2025-01-01 01:52:00,2025-01-01,0.183333,1,1,2,Wednesday,1,...,Lausanne,SN,SN,SBB,2025-01-01 01:00:00,2.6,0.0,0.0,6.2,True
